In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import v2
import torch.optim as optim
from tqdm import tqdm

In [2]:
transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((32, 32)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5,), (0.5,))
])

BATCH_SIZE = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [ ]:
train_dataset = torchvision.datasets.MNIST(root='data/', train=True,
                                           download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE,
                                           shuffle=True, num_workers=2)

test_dataset = torchvision.datasets.MNIST(root='data/', train=True,
                                          download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE,
                                          shuffle=False, num_workers=2)

In [4]:
### LeNet 5

class LeNet(nn.Module):
    
        def __init__(self):
            super(LeNet, self).__init__()
            
            self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, padding=0, stride=1)
            self.subsampling1 = nn.AvgPool2d(kernel_size=2, stride=2, padding=0)
            
            self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, padding=0)
            self.subsampling2 = nn.AvgPool2d(kernel_size=2, stride=2, padding=0)
            
            self.conv3 = nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5, stride=1, padding=0)
            self.flatten = nn.Flatten()
            
            self.fc1 = nn.Linear(in_features=120, out_features=84)
            self.fc2 = nn.Linear(in_features=84, out_features=10)
            
            self.tanh = nn.Tanh()
            
        def forward(self, X):
            x = self.tanh(self.subsampling1(self.conv1(X)))
            x = self.tanh(self.subsampling2(self.conv2(x)))
            x = self.flatten(self.tanh(self.conv3(x)))
            x = self.tanh(self.fc1(x))
            x = self.fc2(x)
            return x
        
model = LeNet().to(device)
model

LeNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (subsampling1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (subsampling2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv3): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=120, out_features=84, bias=True)
  (fc2): Linear(in_features=84, out_features=10, bias=True)
  (tanh): Tanh()
)

In [5]:
# Training loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]", leave=True)
    
    for batch_idx, (inputs, labels) in enumerate(loop):
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        avg_loss = running_loss / (batch_idx + 1)
        
        loop.set_postfix(batch_loss = loss.item(), avg_loss = avg_loss)
        
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        eval_loop = tqdm(test_loader, desc='Evaluating', leave=True)
        for inputs, labels in eval_loop:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            current_acc = 100 * correct / total
            eval_loop.set_postfix(accuracy=f"{current_acc:.2f}%")
            
        print(f"Accuracy: {100 * correct / total:.2f}%")

Evaluating: 100%|██████████| 938/938 [00:16<00:00, 55.23it/s, accuracy=37.18%] 


Accuracy: 37.18%


Evaluating: 100%|██████████| 938/938 [00:17<00:00, 52.70it/s, accuracy=46.73%]


Accuracy: 46.73%


Evaluating: 100%|██████████| 938/938 [00:26<00:00, 35.68it/s, accuracy=54.50%]


...

Accuracy: 83.16%


Evaluating: 100%|██████████| 938/938 [00:12<00:00, 77.55it/s, accuracy=84.61%] 


Accuracy: 84.61%


Evaluating: 100%|██████████| 938/938 [00:12<00:00, 77.78it/s, accuracy=85.69%] 

Accuracy: 85.69%
